# scSLIDE: a synthetic multi-sample example

Draft, not yet executed against real OmicVerse/scslide-py in this authoring environment. No biological discovery is claimed. Install the approved backend and the code PR before running all cells in a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad
import scanpy as sc
import omicverse as ov
from pathlib import Path
import importlib.metadata as metadata

print("omicverse", metadata.version("omicverse"))
print("scslide-py", metadata.version("scslide-py"))

In [ ]:
rng = np.random.default_rng(17)
sample_index = np.repeat(np.arange(12), 30)
counts = rng.poisson(2.0, (360, 40)).astype(float)
counts[:, :8] += rng.poisson((sample_index / 3 + 1)[:, None], (360, 8))
obs = pd.DataFrame({
    "sample": [f"s{i}" for i in sample_index],
    "subject": [f"p{i}" for i in sample_index],
    "condition": sample_index.astype(float),
}, index=[f"c{i}" for i in range(360)])
adata = ad.AnnData(sparse.csr_matrix(counts), obs=obs,
    var=pd.DataFrame(index=[f"g{i}" for i in range(40)]))
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["data"] = adata.X.copy()
# All genes are explicitly selected for this small synthetic example.
adata.var["highly_variable"] = True
sc.pp.pca(adata, n_comps=5, random_state=17)

In [ ]:
result = ov.single.scslide(
    adata, sample_key="sample", response_key="condition", subject_key="subject",
    counts_layer="counts", data_layer="data", embedding_key="X_pca",
    n_jobs=1, random_state=17,
    workflow_kwargs={"ncells_per_group":15, "ncells_landmark":60,
        "ncomp_pls":3, "k_nn":3, "knn_range":30,
        "ncomp_diffusion":3, "fit_curve":False},
)
sample_adata = result.sample_adata
assert sample_adata.n_obs == 12
assert sample_adata.n_vars == 60
assert "scslide" not in adata.uns
print(sample_adata)
print(result.inference_summary)

In [ ]:
sc.pl.embedding(sample_adata, basis="scslide_diffmap", color="condition")

Each point above is a sample, not a cell. The response helped construct the representation, so a color gradient is not independent predictive evidence. No trajectory direction is automatically a disease direction.

In [ ]:
out = Path("scslide_tutorial_outputs")
out.mkdir(exist_ok=False)
sample_adata.write_h5ad(out / "samples.h5ad")
restored = ad.read_h5ad(out / "samples.h5ad")
assert restored.obs_names.equals(sample_adata.obs_names)
assert restored.var_names.equals(sample_adata.var_names)
np.testing.assert_allclose(restored.obsm["X_scslide_diffmap"], sample_adata.obsm["X_scslide_diffmap"])
result.diffusion.embeddings.to_csv(out / "sample_diffusion.csv")
result.sample_object.counts.to_csv(out / "landmark_by_sample_counts.csv")